In [1]:
import csv
import json
import os
import random
import re
from copy import deepcopy
from typing import Callable, Optional, Any

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# Единая реализация модуля

Реализация подмен находится в следующих ячейках. Дублирующая самостоятельная версия удалена.

Эксперимент проверяет чувствительность ответа к предоставленному тексту цепочки; он не раскрывает внутренние вычисления модели.


In [2]:
# Числовые значения и извлечение ответов
import re
import json
import random
from copy import deepcopy

NUMBER_PATTERN = re.compile(
    r"(?<![\w.,])-?\d{1,3}(?:,\d{3})+(?:\.\d+)?(?!\w)"
    r"|(?<![\w.,])-?\d+(?:\.\d+)?(?!\w)"
)
STEP_LABEL = re.compile(r"^\s*(?:step\s*\d+\s*[:.)-]?\s*|\d+[.)]\s+)", re.I)
ANSWER_CUE = "Answer:"
ANSWER_MAX_TOKENS = 24


def find_numbers(step):
    """Числа шага без номера самого шага."""
    label = STEP_LABEL.match(str(step))
    start = label.end() if label else 0
    return [m for m in NUMBER_PATTERN.finditer(str(step)) if m.start() >= start]


def extract_number_ex(text):
    m = re.search(r"Answer:\s*\$?(-?[\d,]*\.?\d+)", str(text))
    if m:
        s, ok = m.group(1), True
    else:
        nums = re.findall(r"-?\d[\d,]*\.?\d*", str(text))
        s, ok = (nums[-1] if nums else None), False
    return (s.replace(",", "").rstrip("."), ok) if s else (None, False)


def extract_letter_ex(text):
    text = str(text)
    m = re.search(r"Answer:\s*\(?([A-E])\b", text, flags=re.I)
    if m:
        return m.group(1).upper(), True
    cands = re.findall(r"(?i:answer\s+is)\s*:?\s*\(?([A-E])\b", text)
    if not cands:
        cands = re.findall(r"\(([A-E])\)", text)
    return (cands[-1].upper(), False) if cands else (None, False)


def same_number(a, b):
    try:
        return float(a) == float(b)
    except (TypeError, ValueError):
        return False


def normalize_dataset(dataset):
    value = str(dataset).strip().lower()
    if value in {"gsm8k", "gsm-8k"}:
        return "gsm8k"
    if value.startswith("arc"):
        return "arc"
    return value


def extract_answer_ex(text, dataset):
    return extract_number_ex(text) if normalize_dataset(dataset) == "gsm8k" else extract_letter_ex(text)


def answers_equal(prediction, reference, dataset):
    if prediction is None or reference is None:
        return False
    if normalize_dataset(dataset) == "gsm8k":
        return same_number(prediction, reference)
    return str(prediction).strip().upper() == str(reference).strip().upper()


def split_steps(text):
    lines = [s.strip() for s in str(text).split("\n\n") if s.strip()]
    return [s for s in lines if not s.startswith("Answer:")]


In [3]:
def parse_steps(value):
    """Читает steps как JSON-список; при старом текстовом формате делит по абзацам."""
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if value is None or not str(value).strip():
        return []
    value = str(value).strip()
    try:
        parsed = json.loads(value)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except json.JSONDecodeError:
        pass
    return split_steps(value)


def serialize_steps(steps):
    return json.dumps(list(steps), ensure_ascii=False)


def extract_question_text(row):
    """Берёт только отдельное поле условия, никогда не сканирует весь few-shot prompt."""
    for key in ("question_text", "question", "problem", "query"):
        value = row.get(key)
        if value is not None and str(value).strip():
            return str(value).strip(), key
    return None, None


def question_numbers_from_text(question_text):
    """Числа только из явно выделенного текста текущей задачи."""
    if question_text is None:
        return None
    values = set()
    for match in NUMBER_PATTERN.finditer(str(question_text)):
        try:
            values.add(float(match.group().replace(",", "")))
        except ValueError:
            continue
    return values


def _format_like(old_text, value):
    decimals = len(old_text.split(".")[1]) if "." in old_text else 0
    return f"{value:,.{decimals}f}" if "," in old_text else f"{value:.{decimals}f}"


def corrupt_value(old_value, decimals, rng):
    unit = 10 ** (-decimals)
    if old_value == 0:
        return rng.choice([1, 2, 3, 5]) * unit
    for _ in range(40):
        fraction = rng.choice([0.1, 0.25, 0.5])
        delta = rng.choice([-1, 1]) * max(unit, abs(old_value) * fraction)
        candidate = round(old_value + delta, decimals)
        if candidate != old_value and candidate * old_value > 0:
            return candidate
    return None


def replace_number(steps, step_index, number_index, rng):
    if not 0 <= step_index < len(steps):
        return None
    matches = find_numbers(steps[step_index])
    if not 0 <= number_index < len(matches):
        return None
    match = matches[number_index]
    old_text = match.group()
    try:
        old_value = float(old_text.replace(",", ""))
    except ValueError:
        return None
    decimals = len(old_text.split(".")[1]) if "." in old_text else 0
    new_value = corrupt_value(old_value, decimals, rng)
    if new_value is None:
        return None
    new_text = _format_like(old_text, new_value)
    if new_text == old_text:
        return None
    modified = list(steps)
    modified[step_index] = steps[step_index][:match.start()] + new_text + steps[step_index][match.end():]
    shift = (new_value - old_value) / old_value if old_value != 0 else None
    return modified, old_text, new_text, shift


def truncate_steps(steps, keep_count):
    """Оставляет ровно первые keep_count шагов; допустимо 0 <= keep_count < n."""
    if not 0 <= keep_count < len(steps):
        raise ValueError("keep_count must be in [0, len(steps)-1]")
    return list(steps[:keep_count])


def swap_adjacent(steps, index):
    if not 0 <= index < len(steps) - 1:
        return None
    changed = list(steps)
    changed[index], changed[index + 1] = changed[index + 1], changed[index]
    if changed == list(steps):
        return None
    return changed


def distinct_permutations(steps, rng, n_permutations=3, max_tries=500):
    """Возвращает уникальные неидентичные перестановки (с сохранением всех шагов)."""
    original = list(steps)
    seen = {tuple(original)}
    result = []
    for _ in range(max_tries):
        order = list(range(len(original)))
        rng.shuffle(order)
        candidate = [original[i] for i in order]
        key = tuple(candidate)
        if key not in seen and candidate != original:
            seen.add(key)
            result.append((order, candidate))
            if len(result) >= n_permutations:
                break
    return result


def relative_position(index, n_steps):
    return index / (n_steps - 1) if n_steps > 1 else 0.0


def build_variants(steps, rng, given_numbers=None, max_numbers=None, n_permutations=3):
    """Все варианты подаются в одном режиме: продолжить после полной цепочки и Answer:."""
    steps = list(steps)
    n = len(steps)
    if n < 2:
        return []
    variants = []

    def add(kind, changed, **meta):
        variants.append({"perturbation": kind, "modified_steps": list(changed), **meta})

    # Один контроль на вопрос: полная исходная цепочка без вмешательства.
    add("control", steps, target_step_index=None, relative_position=None,
        kept_fraction=1.0, removed_step_count=0, permutation_order=None)

    targets = [(si, ni) for si, step in enumerate(steps) for ni in range(len(find_numbers(step)))]
    if max_numbers is not None and len(targets) > max_numbers:
        targets = sorted(rng.sample(targets, max_numbers))
    given = set(given_numbers) if given_numbers is not None else None
    for si, ni in targets:
        result = replace_number(steps, si, ni, rng)
        if result is None:
            continue
        modified, old, new, shift = result
        old_value = float(old.replace(",", ""))
        nums_in_step = find_numbers(steps[si])
        add("number_substitution", modified,
            target_step_index=si, relative_position=relative_position(si, n),
            kept_fraction=1.0, removed_step_count=0,
            target_number_index=ni, old_number=old, new_number=new,
            relative_shift=shift,
            number_is_given=(old_value in given) if given is not None else None,
            is_last_number_in_step=(ni == len(nums_in_step) - 1),
            permutation_order=None)

    # Префиксная обрезка: сохраняются первые k шагов, k=0,...,n-1.
    for keep in range(n):
        add("truncation", truncate_steps(steps, keep),
            target_step_index=None, relative_position=None,
            kept_fraction=keep / n, removed_step_count=n - keep,
            truncation_boundary=keep, permutation_order=None)

    # Перестановки соседей; target_step_index — левая позиция обмена.
    for i in range(n - 1):
        changed = swap_adjacent(steps, i)
        if changed is not None:
            add("adjacent_swap", changed, target_step_index=i,
                relative_position=relative_position(i, n), kept_fraction=1.0,
                removed_step_count=0, permutation_order=None)

    # Случайные полные перестановки с сохранением всех исходных шагов.
    for order, changed in distinct_permutations(steps, rng, n_permutations):
        add("step_permutation", changed, target_step_index=None,
            relative_position=None, kept_fraction=1.0, removed_step_count=0,
            permutation_order=json.dumps(order))

    return variants


def build_assistant_prefix(modified_steps):
    chain = "\n\n".join(modified_steps)
    return (chain + "\n\n" + ANSWER_CUE) if chain else ANSWER_CUE


In [4]:
FIELDNAMES = [
    "question_id", "dataset", "model", "mode",
    "perturbation_id", "perturbation", "n_steps",
    "target_step_index", "relative_position", "kept_fraction", "removed_step_count",
    "truncation_boundary", "target_number_index", "old_number", "new_number", "relative_shift",
    "number_is_given", "is_last_number_in_step", "permutation_order",
    "question_text_source", "original_steps", "perturbed_steps", "assistant_prefix",
    "original_answer", "gold_answer", "regenerated_text", "regenerated_answer",
    "parse_success", "format_ok", "answer_change_status", "answer_changed",
    "regenerated_correct", "original_correct", "control_answer_changed", "control_adjusted_change",
]


def _row_key(row):
    return (str(row.get("question_id", "")), str(row.get("dataset", "")),
            str(row.get("model", "")), str(row.get("mode", "")))


def _truthy(value):
    return str(value).strip().lower() in {"1", "true", "yes", "y"}


def _answer_change(prediction, original, dataset):
    if original is None or not str(original).strip():
        return None, "missing_original"
    if prediction is None or not str(prediction).strip():
        return None, "unparseable_regenerated"
    changed = not answers_equal(prediction, original, dataset)
    return bool(changed), "changed" if changed else "unchanged"


def run_experiment(input_csv, output_csv, generate_fn: Callable,
                   seed=42, max_rows=None, save_every=10, append_existing=False,
                   skip_truncated=True, only_correct=True, max_numbers=None,
                   n_permutations=3):
    """Единый answer-only эксперимент; контроль и все вмешательства парны по вопросу."""
    rng = random.Random(seed)
    tasks = []
    with open(input_csv, "r", encoding="utf-8-sig", newline="") as f:
        for row_number, row in enumerate(csv.DictReader(f)):
            if max_rows is not None and row_number >= max_rows:
                break
            original_steps = parse_steps(row.get("steps", ""))
            if len(original_steps) < 2:
                continue
            if skip_truncated and _truthy(row.get("truncated", "")):
                continue
            if only_correct and not _truthy(row.get("correct", "")):
                continue
            question_text, question_text_source = extract_question_text(row)
            given_numbers = question_numbers_from_text(question_text)
            variants = build_variants(original_steps, rng, given_numbers,
                                      max_numbers=max_numbers, n_permutations=n_permutations)
            tasks.append((row, original_steps, variants, question_text_source))

    append = append_existing and os.path.exists(output_csv) and os.path.getsize(output_csv) > 0
    done = set()
    if append:
        with open(output_csv, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f)
            if reader.fieldnames != FIELDNAMES:
                raise ValueError("Схема CSV изменилась; используйте новый output_csv или append_existing=False.")
            done = {_row_key(r) for r in reader}
    todo = [t for t in tasks if _row_key(t[0]) not in done]
    total = sum(len(v) for _, _, v, _ in todo)
    print(f"Вопросов: {len(tasks)}, уже сохранено: {len(tasks)-len(todo)}, к обработке: {len(todo)}, генераций: {total}")

    out = open(output_csv, "a" if append else "w",
               encoding="utf-8" if append else "utf-8-sig", newline="")
    writer = csv.DictWriter(out, fieldnames=FIELDNAMES)
    if not append:
        writer.writeheader(); out.flush()
    pending, results = [], []
    saved = 0
    bar = tqdm(total=total, desc="Генерация", unit="ген")

    def flush():
        nonlocal saved
        if pending:
            writer.writerows(pending); out.flush(); os.fsync(out.fileno())
            saved += len(pending); pending.clear()

    try:
        for task_idx, (row, original_steps, variants, question_text_source) in enumerate(todo, 1):
            dataset = normalize_dataset(row.get("dataset", ""))
            original_answer = row.get("extracted_answer")
            gold_answer = row.get("gold_answer")
            question_results = []
            for variant_id, variant in enumerate(variants):
                prefix = build_assistant_prefix(variant["modified_steps"])
                generated_text = generate_fn(row.get("prompt", ""), row, prefix, ANSWER_MAX_TOKENS)
                prediction, format_ok = extract_answer_ex(ANSWER_CUE + str(generated_text), dataset)
                parse_success = prediction is not None and str(prediction).strip() != ""
                answer_changed, change_status = _answer_change(prediction, original_answer, dataset)
                regenerated_correct = (answers_equal(prediction, gold_answer, dataset)
                                        if prediction is not None and gold_answer is not None else None)
                record = {k: row.get(k) for k in ("question_id", "dataset", "model", "mode")}
                record.update({
                    "perturbation_id": variant_id, "perturbation": variant["perturbation"],
                    "n_steps": len(original_steps),
                    "target_step_index": variant.get("target_step_index"),
                    "relative_position": variant.get("relative_position"),
                    "kept_fraction": variant.get("kept_fraction"),
                    "removed_step_count": variant.get("removed_step_count"),
                    "truncation_boundary": variant.get("truncation_boundary"),
                    "target_number_index": variant.get("target_number_index"),
                    "old_number": variant.get("old_number"), "new_number": variant.get("new_number"),
                    "relative_shift": variant.get("relative_shift"),
                    "number_is_given": variant.get("number_is_given"),
                    "is_last_number_in_step": variant.get("is_last_number_in_step"),
                    "permutation_order": variant.get("permutation_order"),
                    "question_text_source": question_text_source,
                    "original_steps": serialize_steps(original_steps),
                    "perturbed_steps": serialize_steps(variant["modified_steps"]),
                    "assistant_prefix": prefix, "original_answer": original_answer,
                    "gold_answer": gold_answer, "regenerated_text": generated_text,
                    "regenerated_answer": prediction, "parse_success": parse_success,
                    "format_ok": bool(format_ok), "answer_change_status": change_status,
                    "answer_changed": answer_changed, "regenerated_correct": regenerated_correct,
                    "original_correct": _truthy(row.get("correct", "")),
                    "control_answer_changed": None, "control_adjusted_change": None,
                })
                question_results.append(record); bar.update(1)
            # Контроль один на вопрос. Парное сравнение каждого вмешательства с ним.
            control = next((x for x in question_results if x["perturbation"] == "control"), None)
            control_value = control["answer_changed"] if control else None
            for record in question_results:
                if record["perturbation"] != "control":
                    record["control_answer_changed"] = control_value
                    if record["answer_changed"] is not None and control_value is not None:
                        record["control_adjusted_change"] = int(record["answer_changed"]) - int(control_value)
            pending.extend(question_results); results.extend(question_results)
            if task_idx % max(1, save_every) == 0:
                flush()
    finally:
        flush(); bar.close(); out.close()
    print(f"Готово. Сохранено вариантов: {len(results)}; файл: {output_csv}")
    return results


Устаревшая дублирующая реализация удалена. Используется единый модуль в ячейках 2–4.

Устаревшая дублирующая реализация run_experiment удалена. Используется runner из ячейки 4.

In [5]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Устройство:", model.device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Устройство: cuda:0


In [6]:
def generate(model, tokenizer, user_prompt, max_new_tokens, assistant_prefix=""):
    """assistant_prefix - начало ответа ассистента, которое модель продолжает."""
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    text += assistant_prefix
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                temperature=None, top_p=None, top_k=None)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), len(new_tokens)

In [7]:
def qwen_generate(prompt, row, assistant_prefix="", max_new_tokens=700):
    generated_text, n_tokens = generate(
        model,
        tokenizer,
        prompt,
        max_new_tokens=max_new_tokens,
        assistant_prefix=assistant_prefix,
    )
    return generated_text

In [ ]:
import pandas as pd

#INPUT_CSV = "/kaggle/input/datasets/yumiasakura/ml-cw-qwen2505/results_test100_Qwen2.5-0.5B-Instruct.csv"
INPUT_CSV = '/kaggle/input/datasets/yumiasakura/result-test100-qwen2505-new-columns/results_test100_Qwen2.5-0.5B-Instruct_with_question_text.csv'
OUTPUT_CSV = "/kaggle/working/perturbation_results_0.5B.csv"
'''
OUTPUT_CSV = "/kaggle/working/perturbation_results_0.5B.csv"


df = pd.read_csv("/kaggle/input/datasets/yumiasakura/ml-cw-qwen2505/results_test100_Qwen2.5-0.5B-Instruct.csv")
df_trimmed = df.sample(n=30)

df_trimmed.to_csv('/kaggle/working/output_file.csv', index=False)

print("Файл успешно обрезан и сохранен!")
INPUT_CSV = '/kaggle/working/output_file.csv'
'''
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)


run_experiment(
    input_csv=INPUT_CSV,
    output_csv=OUTPUT_CSV,
    generate_fn=qwen_generate,
    seed=42,
    save_every=1,
    append_existing=False, # True - дописывать после падения, не пересчитывая сохранённые вопросы
    skip_truncated=True, # не брать цепочки, обрезанные по лимиту токенов
    only_correct=True,   # брать только изначально верные ответы
    max_numbers=None,    # например 12 - ограничить число подменяемых чисел на вопрос
)

Вопросов: 148, уже сохранено: 0, к обработке: 148, генераций: 3802


Генерация:   0%|          | 0/3802 [00:00<?, ?ген/s]

Готово. Сохранено вариантов: 3802; файл: /kaggle/working/perturbation_results_0.5B.csv


In [ ]:
import numpy as np
import pandas as pd


def _to_bool_or_nan(value):
    if pd.isna(value): return np.nan
    s = str(value).strip().lower()
    if s in {"true", "1", "yes"}: return 1.0
    if s in {"false", "0", "no"}: return 0.0
    return np.nan


def load_results(path):
    frame = pd.read_csv(path, encoding="utf-8-sig")
    for col in ("answer_changed", "regenerated_correct", "format_ok", "parse_success",
                "original_correct", "number_is_given", "is_last_number_in_step",
                "control_answer_changed", "control_adjusted_change"):
        if col in frame:
            frame[col] = frame[col].map(_to_bool_or_nan)
    frame["position"] = frame["relative_position"]
    frame.loc[frame["perturbation"] == "truncation", "position"] = frame.loc[
        frame["perturbation"] == "truncation", "kept_fraction"]
    return frame


def bootstrap_mean(values, n_boot=2000, seed=0):
    x = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(x) == 0: return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = rng.choice(x, size=(n_boot, len(x)), replace=True).mean(axis=1)
    return float(x.mean()), *np.percentile(means, [2.5, 97.5])


def summarize(frame, by, value="answer_changed", n_boot=2000):
    """Бутстрэп по вопросам; не фильтрует строки по format_ok."""
    by = list(by)
    keys = list(dict.fromkeys(by + ["question_id", "dataset", "model", "mode"]))
    per_question = frame.groupby(keys, dropna=False, observed=True)[value].mean().reset_index()
    rows = []
    for key, group in per_question.groupby(by, dropna=False, observed=True):
        key = key if isinstance(key, tuple) else (key,)
        mean, lo, hi = bootstrap_mean(group[value], n_boot=n_boot)
        rows.append({**dict(zip(by, key)), "n_questions": group[value].notna().sum(),
                     value: mean, "ci_lo": lo, "ci_hi": hi})
    return pd.DataFrame(rows)


df = load_results(OUTPUT_CSV)
print("Строк:", len(df), "| вопросов:", df["question_id"].nunique())
print("Доля извлечённых ответов:", round(df["parse_success"].mean(), 3))
print("Доля строгого формата Answer:", round(df["format_ok"].mean(), 3))


In [ ]:
# Основные метрики. Не отбрасываем format_ok=False: fallback-ответы учитываются,
# а случаи без извлечённого ответа отдельно видны через parse_success.
summary = summarize(df, ["dataset", "mode", "perturbation"], "answer_changed")
parse_summary = df.groupby(["dataset", "mode", "perturbation"], dropna=False).agg(
    n_trials=("question_id", "size"), parse_success_rate=("parse_success", "mean"),
    strict_format_rate=("format_ok", "mean"),
    regenerated_accuracy=("regenerated_correct", "mean")
).reset_index()
summary.merge(parse_summary, on=["dataset", "mode", "perturbation"], how="left")


In [ ]:
# Парное сравнение каждого вмешательства с неизменённым контролем того же вопроса.
# control_adjusted_change = changed(perturbed) - changed(control), диапазон [-1, 1].
paired = df[df["perturbation"] != "control"].copy()
paired_ci = summarize(paired, ["dataset", "mode", "perturbation"], "control_adjusted_change")
paired_counts = paired.groupby(["dataset", "mode", "perturbation"], dropna=False).agg(
    n_trials=("question_id", "size"),
    perturbed_change_rate=("answer_changed", "mean"),
    control_change_rate=("control_answer_changed", "mean"),
    parse_success_rate=("parse_success", "mean")
).reset_index()
paired_ci.merge(paired_counts, on=["dataset", "mode", "perturbation"], how="left")


In [ ]:
# Позиционные эффекты: позиция шага и доля сохранённой цепочки анализируются раздельно.
numsub = df[df["perturbation"] == "number_substitution"]
summarize(numsub, ["dataset", "mode", "relative_position", "number_is_given", "is_last_number_in_step"])


In [ ]:
# Расширенные метрики: модель, размер модели, тип задачи и позиция вмешательства.
# Для размера модели желательно задать явный столбец model_size в исходном CSV
# (например: 0.5B, 3B, 7B). Автоматическая эвристика ниже — только fallback.
import re
import numpy as np
import pandas as pd

def infer_model_size(model_name):
    match = re.search(r"(\d+(?:\.\d+)?)\s*B\b", str(model_name), flags=re.I)
    return match.group(1) + "B" if match else "unknown"

def prepare_extended_metrics(frame):
    data = frame.copy()
    if "task_type" not in data.columns:
        # Датасет используется как приближение типа задачи, если отдельного поля нет.
        data["task_type"] = data["dataset"]
    if "model_size" not in data.columns:
        data["model_size"] = data["model"].map(infer_model_size)
    data["position"] = data["relative_position"].fillna(data["kept_fraction"])
    data["position_band"] = pd.cut(
        data["position"], bins=[-0.001, 1/3, 2/3, 1.001],
        labels=["начало", "середина", "конец"], include_lowest=True
    )
    return data

def metric_table(frame, group_cols):
    """Усреднение сначала по вопросу, затем по группам; единица бутстрэпа — вопрос."""
    cols = list(dict.fromkeys(group_cols + ["question_id", "model", "mode"]))
    per_question = frame.groupby(cols, dropna=False, observed=True).agg(
        answer_changed=("answer_changed", "mean"),
        regenerated_correct=("regenerated_correct", "mean")
    ).reset_index()
    return per_question.groupby(group_cols, dropna=False, observed=True).agg(
        n_questions=("question_id", "nunique"),
        answer_changed_rate=("answer_changed", "mean"),
        regenerated_accuracy=("regenerated_correct", "mean")
    ).reset_index()

extended = prepare_extended_metrics(df)

# 1. Общая устойчивость: тип вмешательства × тип задачи × размер модели.
metrics_by_model_task = metric_table(
    extended, ["model", "model_size", "task_type", "perturbation"]
)
display(metrics_by_model_task.sort_values(
    ["task_type", "model_size", "perturbation"]
))

# 2. Зависимость от позиции вмешательства.
metrics_by_position = metric_table(
    extended[extended["perturbation"].isin(
        ["number_substitution", "adjacent_swap", "truncation"]
    )],
    ["model_size", "task_type", "perturbation", "position_band"]
)
display(metrics_by_position.sort_values(
    ["task_type", "model_size", "perturbation", "position_band"]
))

# 3. Отдельная сводка: насколько вмешательство ухудшило точность относительно контроля.
control = metric_table(
    extended[extended["perturbation"] == "control"],
    ["model", "task_type", "perturbation"]
)
display(control)


## Интерпретация и ограничения

- Все варианты генерируются в одном режиме `answer`: модель получает полную изменённую цепочку и продолжает после `Answer:`.
- `answer_changed` сравнивает извлечённый ответ с исходным ответом. Если исходный или новый ответ отсутствует, значение остаётся пропущенным, а причина указана в `answer_change_status`.
- `parse_success` и `format_ok` показывают соответственно успешность извлечения и соблюдение строгого формата. Строки с `format_ok=False` не удаляются из основного анализа.
- `control_adjusted_change` — парная разность индикаторов изменения ответа между вмешательством и контролем того же вопроса.
- Для чисел из условия используются только отдельные поля `question_text`, `question`, `problem` или `query`. Если такого поля нет, `number_is_given` остаётся неизвестным; полный few-shot prompt не сканируется.
- Для обрезки `kept_fraction` — доля сохранённых первых шагов. Для замен и соседних перестановок `relative_position` — нормированная позиция шага.
- Эксперимент измеряет чувствительность ответа к переданному тексту цепочки, но не доказывает наличие или использование внутреннего CoT.


## Проверка корректности модуля подмены

1. **Инварианты на уровне строк**: каждая подмена должна менять цепочку ровно так, как заявлено.
2. **format_ok**: сколько вариантов выпадает из таблиц из-за формата ответа и не смещает ли это оценки.
3. **control**: воспроизводит ли продолжение немодифицированного префикса исходный хвост цепочки.
4. **Эвристика number_is_given**: не захватывает ли она числа из few-shot примеров.

In [ ]:
# Проверка инвариантов каждого вида вмешательства.
raw = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig",
                  dtype={"old_number": str, "new_number": str, "question_id": str})
raw["orig"] = raw["original_steps"].map(json.loads)
raw["pert"] = raw["perturbed_steps"].map(json.loads)


def _num(value):
    return float(str(value).replace(",", ""))


def check_row(r):
    o, p, kind = r["orig"], r["pert"], r["perturbation"]
    n = len(o)
    if kind == "control":
        return p == o
    if kind == "truncation":
        keep = int(r["truncation_boundary"])
        return p == o[:keep] and len(p) == keep and keep < n
    if kind == "adjacent_swap":
        i = int(r["target_step_index"])
        if not 0 <= i < n-1: return False
        expected = list(o); expected[i], expected[i+1] = expected[i+1], expected[i]
        return p == expected and p != o
    if kind == "step_permutation":
        try: order = json.loads(r["permutation_order"])
        except Exception: return False
        return (sorted(order) == list(range(n)) and p == [o[i] for i in order]
                and p != o and sorted(p) == sorted(o))
    if kind == "number_substitution":
        i, k = int(r["target_step_index"]), int(r["target_number_index"])
        if len(p) != n or any(p[j] != o[j] for j in range(n) if j != i): return False
        matches = find_numbers(o[i])
        if not 0 <= k < len(matches): return False
        m = matches[k]; old, new = str(r["old_number"]), str(r["new_number"])
        same_sign = _num(old) == 0 or _num(old) * _num(new) > 0
        expected = o[i][:m.start()] + new + o[i][m.end():]
        return m.group() == old and new != old and same_sign and p[i] == expected
    return False

raw["invariant_ok"] = raw.apply(check_row, axis=1)
display(raw.groupby("perturbation")["invariant_ok"].agg(rows="size", share_ok="mean"))


In [ ]:
# Доля успешного извлечения и строгого формата по видам вмешательства.
fmt = df.groupby(["dataset", "mode", "perturbation"], dropna=False).agg(
    rows=("question_id", "size"), parse_success_rate=("parse_success", "mean"),
    strict_format_rate=("format_ok", "mean"),
    unparseable=("answer_change_status", lambda s: (s == "unparseable_regenerated").sum())
).reset_index()
display(fmt)


In [ ]:
# Проверка парности: на каждый вопрос должен приходиться ровно один контроль.
control_counts = raw[raw["perturbation"] == "control"].groupby(
    ["question_id", "dataset", "model", "mode"]
).size()
print("Контролей на вопрос (распределение):")
display(control_counts.value_counts().rename_axis("n_controls").to_frame("n_questions"))


In [ ]:
# Доступность классификации чисел исходного условия.
ns = df[df["perturbation"] == "number_substitution"]
display(ns.groupby(["dataset", "mode"], dropna=False)["number_is_given"]
          .agg(rows="size", known="count", share_given="mean").round(3))
print("Если known=0, добавьте в исходный CSV отдельный столбец question_text.")
